In [2]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import os
import time

In [3]:
data_path = "D:/Harshita Ajmani/Code_harshu/NLP/data/cleaned_data.csv"
save_path = "D:/Harshita Ajmani/Code_harshu/NLP/data/"
model = "intfloat/multilingual-e5-base"
batch_size = 64

#instead of encoding one record at a time, it processes 64 records simultaneously, which is much faster on a GPU

In [4]:
#Loading data

df = pd.read_csv(data_path)
print(f"Data Loaded {len(df):,} records")


Data Loaded 46,468 records


In [5]:
#Building Searchable Text (Passages)

def build_passage(row, lang):
    title    = row[f'title_{lang}']    or ""
    desc     = row[f'desc_{lang}']     or ""
    keywords = row[f'keywords_{lang}'] or ""
    text = f"{title}. {desc} {keywords}".strip()
    return f"passage: {text}"

df['passage_en'] = df.apply(lambda row: build_passage(row, 'en'), axis=1)
df['passage_fr'] = df.apply(lambda row: build_passage(row, 'fr'), axis=1)

print(f"Sample EN passage:\n   {df['passage_en'].iloc[0][:150]}...")
print(f"Sample FR passage:\n   {df['passage_fr'].iloc[0][:150]}...")

Sample EN passage:
   passage: Principal Mineral Areas, Producing Mines, and Oil and Gas Fields (900A). This dataset is produced and published annually by Natural Resources...
Sample FR passage:
   passage: Principales régions minières, principales mines productrices, principaux champs de pétrole et de gaz (900A). Ce jeu de données est produit et...


In [6]:
# Load the model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device} ({torch.cuda.get_device_name(0)})")    
model = SentenceTransformer(model, device=device)
print("Model loaded successfully.")


Using device: cuda (NVIDIA GeForce RTX 5070 Laptop GPU)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\harsh\miniconda3\envs\nlpsearch\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\harsh\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Model loaded successfully.


In [7]:
#Generating English Embeddings

start = time.time()

embeddings_en = model.encode(
    df['passage_en'].tolist(),
    batch_size = batch_size,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"English embeddings done in {time.time() - start:.1f}s")
print(f"Shape: {embeddings_en.shape}")

# normalize_embeddings=True
# Scales each vector to length 1 — required for cosine similarity to work correctly

Batches:   0%|          | 0/727 [00:00<?, ?it/s]

English embeddings done in 242.6s
Shape: (46468, 768)


In [8]:
# Generating French Embeddings

start = time.time()

embeddings_fr = model.encode(
    df['passage_fr'].tolist(),
    batch_size = batch_size,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f"French embeddings done in {time.time() - start:.1f}s")
print(f"Shape: {embeddings_fr.shape}")


Batches:   0%|          | 0/727 [00:00<?, ?it/s]

French embeddings done in 233.3s
Shape: (46468, 768)


In [ ]:
# Saving the embeddings

np.save(os.path.join(save_path, "embeddings_en.npy"), embeddings_en)
np.save(os.path.join(save_path, "embeddings_fr.npy"), embeddings_fr)

df[['id', 'title_en', 'title_fr', 'desc_en', 'desc_fr', 'keywords_en', 'keywords_fr', 'subject', 'org']].to_csv(os.path.join(save_path, "index_data.csv"), index=False)

print(f"Saved embeddings_en.npy  → {embeddings_en.nbytes / 1e6:.1f} MB")
print(f"Saved embeddings_fr.npy  → {embeddings_fr.nbytes / 1e6:.1f} MB")
print(f"Saved index_data.csv")

Saved embeddings_en.npy  → 142.7 MB
Saved embeddings_fr.npy  → 142.7 MB
Saved index_data.csv
